# 🎬 Wan 2.1 Video Generation on Free Google Colab (NVIDIA T4 16GB)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/wan2.1-t4-colab/blob/main/wan2_1_t4_colab_free_inference.ipynb)
![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)
![Model: Wan 2.1](https://img.shields.io/badge/Model-Wan2.1--1.3B-blue.svg)
![Hardware: Tesla T4](https://img.shields.io/badge/GPU-Tesla%20T4%20(15GB)-green.svg)

Generate high-quality videos using **Alibaba's Wan 2.1 (T2V-1.3B)** on a **100% free Google Colab instance** with an NVIDIA Tesla T4 (15 GB VRAM).

### 🧠 How It Works Under the Hood:
Standard Wan 2.1 generation easily exceeds 16 GB of VRAM due to the 4.8B UMT5-XXL text encoder and 3D spatio-temporal attention. This notebook eliminates OOM errors using three complementary optimizations:
1. **Model CPU Offloading:** The heavy text encoder is loaded, extracts prompt embeddings, and is immediately offloaded to system RAM before the 1.3B DiT starts denoising.
2. **3D VAE Tiling & Slicing:** Decodes latent representations into video frames in temporal slices, eliminating the final VRAM spike.
3. **FP16 Tensor Core Arithmetic:** Tailored specifically for Turing architecture (Tesla T4).

**Peak VRAM Usage:** Drops from >16 GB down to **~7.8 GB**!

## 1. Check GPU Runtime
Ensure you have selected **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Install Required Libraries
Install HuggingFace `diffusers`, `transformers`, `accelerate`, and `gradio`.

In [ ]:
# Fix Colab conflict: uninstall outdated torchao so diffusers uses native PyTorch/bitsandbytes
!pip uninstall -y torchao
!pip install -q -U "diffusers>=0.33.0" "transformers>=4.48.0" "accelerate>=0.34.0" bitsandbytes gradio imageio[ffmpeg] sentencepiece protobuf


## 3. Load the Optimized Wan 2.1 Pipeline
Loads the official `Wan-AI/Wan2.1-T2V-1.3B-Diffusers` model with sequential offload and VAE tiling.

In [ ]:
import gc
import torch

# Prevent Colab torchao import conflict
try:
    import diffusers.utils.import_utils as _iu
    _iu._torchao_available = False
except Exception:
    pass
try:
    import torchao.quantization
    if not hasattr(torchao.quantization, "FqnToConfig"):
        class FqnToConfig:
            def __init__(self, *args, **kwargs): pass
        torchao.quantization.FqnToConfig = FqnToConfig
except Exception:
    pass

from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

clear_gpu_memory()

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
print(f"[1/3] Loading VAE from {model_id}...")
vae = AutoencoderKLWan.from_pretrained(
    model_id,
    subfolder="vae",
    torch_dtype=torch.float32,
)

print(f"[2/3] Loading WanPipeline (FP16)...
(This downloads ~14 GB of weights on first run; takes ~2-3 mins in Colab)")
pipe = WanPipeline.from_pretrained(
    model_id,
    vae=vae,
    torch_dtype=torch.float16,
)

print("[3/3] Enabling Model CPU Offloading & VAE Tiling...")
pipe.enable_model_cpu_offload()
if hasattr(pipe, "vae") and pipe.vae is not None:
    if hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()
    if hasattr(pipe.vae, "enable_slicing"):
        pipe.vae.enable_slicing()

allocated = torch.cuda.memory_allocated() / (1024**3)
print(f"✓ Wan 2.1 Ready! Current GPU Allocated: {allocated:.2f} GB")


## 4. Quick Single Video Generation (Script Mode)
Test direct inference and preview the output video inside the notebook.

In [ ]:
from IPython.display import Video, display

prompt = "A futuristic cybernetic tiger stalking smoothly through neon-lit Tokyo streets at night, reflection in wet pavement, 4k cinematic."
negative_prompt = "Bright, overexposed, static, low quality, distorted, blurry, deformed, flickering"

print("Generating video (approx. 2-3 mins on T4 GPU)...")
clear_gpu_memory()

with torch.inference_mode():
    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=480,
        width=832,         # 16:9 widescreen
        num_frames=33,     # 33 frames ≈ 2 seconds at 16fps
        num_inference_steps=25,
        guidance_scale=5.0,
        generator=torch.Generator(device="cpu").manual_seed(42),
    )

frames = output.frames[0] if hasattr(output, "frames") else output.images[0]
output_file = "demo_output.mp4"
export_to_video(frames, output_file, fps=16)
clear_gpu_memory()

print(f"Video successfully generated: {output_file}")
display(Video(output_file, embed=True, width=700))

## 5. Launch Interactive Gradio Web App with Public Link
Run this cell to launch the web interface. Gradio will provide a public `https://xxxx.gradio.live` link so anyone can generate videos directly from their browser or phone!

In [ ]:
import random
import gradio as gr

def gradio_generate(prompt, neg_prompt, resolution, num_frames, steps, guidance_scale, seed, progress=gr.Progress(track_tqdm=True)):
    if not prompt or not prompt.strip():
        raise gr.Error("Please enter a prompt.")
    
    res_dict = {
        "832x480 (Landscape 16:9)": (832, 480),
        "480x832 (Portrait 9:16)": (480, 832),
        "624x624 (Square 1:1)": (624, 624),
    }
    w, h = res_dict.get(resolution, (832, 480))
    actual_seed = random.randint(0, 2**31 - 1) if seed == -1 else int(seed)
    
    clear_gpu_memory()
    out_path = f"wan_{actual_seed}.mp4"
    
    try:
        progress(0.1, desc="Encoding prompt and running 3D DiT denoising...")
        with torch.inference_mode():
            out = pipe(
                prompt=prompt,
                negative_prompt=neg_prompt,
                height=h,
                width=w,
                num_frames=int(num_frames),
                num_inference_steps=int(steps),
                guidance_scale=float(guidance_scale),
                generator=torch.Generator(device="cpu").manual_seed(actual_seed),
            )
        frames = out.frames[0] if hasattr(out, "frames") else out.images[0]
        export_to_video(frames, out_path, fps=16)
        clear_gpu_memory()
        return out_path, f"✓ Success! Seed: {actual_seed}"
    except Exception as e:
        clear_gpu_memory()
        return None, f"Error: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), title="Wan 2.1 Colab T4") as demo:
    gr.Markdown("# 🎬 Wan 2.1 Video Generator (Free Colab T4 16GB)")
    gr.Markdown("Powered by **Alibaba Wan 2.1 (T2V-1.3B)**. Running with CPU offload and VAE tiling under 8 GB VRAM.")
    
    with gr.Row():
        with gr.Column(scale=5):
            p_in = gr.Textbox(label="Prompt", lines=3, value="A sleek racing car drifting around a sharp mountain curve at sunset, cinematic lighting, tire smoke.")
            with gr.Accordion("Negative Prompt", open=False):
                np_in = gr.Textbox(label="Negative Prompt", lines=2, value="Bright, overexposed, static, low quality, distorted, blurry, deformed")
            
            with gr.Row():
                res_in = gr.Dropdown(label="Resolution", choices=["832x480 (Landscape 16:9)", "480x832 (Portrait 9:16)", "624x624 (Square 1:1)"], value="832x480 (Landscape 16:9)")
                f_in = gr.Slider(label="Frames (33 ≈ 2s)", minimum=17, maximum=49, step=8, value=33)
            with gr.Row():
                s_in = gr.Slider(label="Steps", minimum=15, maximum=35, step=1, value=25)
                g_in = gr.Slider(label="Guidance Scale", minimum=3.0, maximum=8.0, step=0.5, value=5.0)
            with gr.Row():
                seed_in = gr.Number(label="Seed (-1 for random)", value=-1, precision=0)
                rnd_btn = gr.Button("🎲 Random", scale=1)
            rnd_btn.click(lambda: -1, outputs=[seed_in])
            gen_btn = gr.Button("🚀 Generate Video", variant="primary", size="lg")
        
        with gr.Column(scale=5):
            vid_out = gr.Video(label="Generated Video", autoplay=True, loop=True)
            status_out = gr.Markdown("Ready to generate.")
    
    gen_btn.click(fn=gradio_generate, inputs=[p_in, np_in, res_in, f_in, s_in, g_in, seed_in], outputs=[vid_out, status_out])

# Launch public link with share=True
demo.queue().launch(share=True, debug=True)